In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

In [ ]:
cats = ["sci.space", "rec.sport.hockey", "comp.graphics", "talk.politics.mideast"]
rm = ("headers", "footers", "quotes")
train = fetch_20newsgroups(subset="train", categories=cats, remove=rm)
test  = fetch_20newsgroups(subset="test",  categories=cats, remove=rm)
print("Train docs:", len(train.data), "| Test docs:", len(test.data))
print("Classes   :", train.target_names)
print("\nSample:", train.data[0][:200].replace("\n", " "))

In [ ]:
nb  = make_pipeline(TfidfVectorizer(stop_words="english", sublinear_tf=True, max_df=0.8), MultinomialNB(alpha=0.1))
svm = make_pipeline(TfidfVectorizer(stop_words="english", sublinear_tf=True, max_df=0.8), LinearSVC())

nb.fit(train.data, train.target)
svm.fit(train.data, train.target)

In [ ]:
results = {}
for name, m in [("Naive Bayes", nb), ("Linear SVM", svm)]:
    pred = m.predict(test.data)
    results[name] = accuracy_score(test.target, pred)
    print(f"===== {name} | accuracy = {results[name]:.4f} =====")
    print(classification_report(test.target, pred, target_names=train.target_names))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, (name, m) in zip(ax, [("Naive Bayes", nb), ("Linear SVM", svm)]):
    ConfusionMatrixDisplay.from_estimator(m, test.data, test.target, display_labels=train.target_names,
                                          ax=a, xticks_rotation=45, colorbar=False, cmap="Blues")
    a.set_title(name)
plt.tight_layout(); plt.show()

plt.bar(results.keys(), results.values(), color=["orange", "seagreen"]); plt.ylim(0.5, 1)
plt.ylabel("Accuracy"); plt.title("NB vs SVM"); plt.show()

In [ ]:
samples = ["NASA launched a new rocket to the moon",
           "The goalie saved the penalty in the final period",
           "The new GPU renders 3D images much faster"]
for s, p in zip(samples, svm.predict(samples)):
    print(f"{train.target_names[p]:25s} <- {s}")